# Wild-Cluster Bootstrap Inference

**Econometrics Notebook Library · v0.1.0**

## Intuition

Cluster-robust asymptotics rely on the number of independent clusters growing. With few clusters—or few treated clusters—the usual $t$ approximation can badly over-reject. The wild-cluster bootstrap preserves within-cluster dependence by multiplying residuals by a common random weight inside each cluster.

Cameron, Gelbach & Miller show bootstrap-$t$ refinements can improve size with few clusters. MacKinnon & Webb emphasize that very few treated clusters remain a hard case.

References: [Cameron, Gelbach & Miller (2008)](https://doi.org/10.1162/rest.90.3.414) · [MacKinnon & Webb (2018)](https://doi.org/10.1111/ectj.12107).

## Null-imposed bootstrap-$t$

For $H_0:\beta_j=\beta_0$:

1. Fit the restricted model under $H_0$ and obtain restricted residuals $\tilde u_i$.
2. Draw one weight $w_g$ for each cluster $g$.
3. Generate $Y_i^*=\widehat Y_i^{(0)}+w_{g(i)}\tilde u_i$.
4. Refit the unrestricted model and compute $t_b^*=(\hat\beta_{j,b}^*-\beta_0)/SE_b^*$.
5. Compare $|t_{obs}|$ to the empirical distribution of $|t_b^*|$.

Rademacher weights are $\{-1,+1\}$. Webb's six-point weights add support when the number of clusters is small.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

In [ ]:
import statsmodels.api as sm
from scipy.stats import t as t_dist
from econnotes.core import simulate_clustered_regression, wild_cluster_bootstrap_t

df = simulate_clustered_regression(n_clusters=12, cluster_size=45, beta=0.0, seed=73)
X = sm.add_constant(df[["x"]]).to_numpy()
fit = sm.OLS(df.y, X).fit(cov_type="cluster", cov_kwds={"groups":df.cluster})
beta_hat = float(np.asarray(fit.params)[1])
cluster_se = float(np.asarray(fit.bse)[1])
naive_t = beta_hat / cluster_se
naive_p = 2*(1-t_dist.cdf(abs(naive_t), df=df.cluster.nunique()-1))
boot = wild_cluster_bootstrap_t(df.y.to_numpy(), X, df.cluster.to_numpy(), tested_index=1, B=499, weights="webb", seed=7)
{"beta":beta_hat, "cluster_se":cluster_se, "t":naive_t, "cluster_t_p":naive_p, "wild_p":boot["p_wild"]}

In [ ]:
fig, ax = plt.subplots()
ax.hist(boot["t_boot"], bins=35, density=True, alpha=.7)
ax.axvline(boot["t_obs"], linestyle="--", label="Observed t")
ax.axvline(-boot["t_obs"], linestyle="--")
ax.set(xlabel="Bootstrap t statistic", ylabel="Density", title="Null-imposed wild-cluster bootstrap distribution")
ax.legend();

## Common failure

The wild bootstrap does not rescue designs with effectively no independent treated variation. If treatment occurs in two clusters, there are only a tiny number of meaningful assignment patterns. Resampling sophistication cannot manufacture identifying variation.

## Researcher failure checklist

- Cluster at the level of treatment assignment or residual dependence justified by the design.
- Report the number of clusters **and treated clusters**.
- Impose the null when constructing bootstrap residuals for hypothesis testing.
- Use Webb or related weights thoughtfully with few clusters; do not treat weight choice as cosmetic.
- Consider randomization/placebo inference when treatment assignment has a known cluster-level mechanism.